# **Introduction to LangGraph**
## **What's Covered?**
## **What is LangGraph?**
## **Why is LangGraph powerful?**


## **Key Components of LangGraph**
### **1. State and Reducer**


In [ ]:
# State and Reducer
# TemperatureState is the state which will be shared with the entire workflow/graph
# Reducer helps us define how the updates are to be implemented on each state property
# If we donot explicity bind the reducer with each property - by default overwrite (replace)

from typing import TypedDict

# Define a State
class TemperatureState(TypedDict):
    celsius: float      # Input
    fahrenheit: float   # Output

### **2. Defining a Graph (Nodes and Edges)**
#### **Nodes**


In [ ]:
d = {"a" : 1, "b" : 2}

In [ ]:
d["a"]

In [ ]:
def celsius2fahrenheit_converter(state: TemperatureState) -> TemperatureState:
    # 1. READ the input
    c_temp = state["celsius"]
    
    # 2. CALCULATE
    # Formula: (C * 9/5) + 32
    f_temp = (c_temp * 1.8) + 32

    # 3. WRITE the update
    state["fahrenheit"] = round(f_temp, 2)
    
    # 3. WRITE the update
    return state

In [ ]:
# def celsius2fahrenheit_converter(state: TemperatureState) -> TemperatureState:
#     # 1. READ the input
#     c_temp = state["celsius"]
    
#     # 2. CALCULATE
#     # Formula: (C * 9/5) + 32
#     f_temp = (c_temp * 1.8) + 32
    
#     # 3. WRITE the update
#     return {"fahrenheit" : round(f_temp, 2)}

#### **Edges**


In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(TemperatureState)

# Add Nodes
graph.add_node("c2f_node", celsius2fahrenheit_converter)

# Add Edges
graph.add_edge(START, "c2f_node")
graph.add_edge("c2f_node", END)

### **3. Compiling a StateGraph**


In [ ]:
workflow = graph.compile()

### **4. Invoking the Workflow**

In [ ]:
workflow.invoke({"celsius": 5})

### **Visualizing the Graph**
# from langchain_core.runnables.graph import CurveStyle, MermaidDrawMethod, NodeStyles


In [ ]:
from IPython.display import Image, display

display(Image(workflow.get_graph().draw_mermaid_png()))

## **Graph with Multiple Inputs**

In [ ]:
# Step 1: Defining a State
from typing import TypedDict, List

# Define a typed state
class AgentState(TypedDict):
    name: str
    values: List[int]
    response: str

In [ ]:
# Step 2: Define a node

def processor_node(state: AgentState) -> AgentState:
    """This function takes a list and return the sum of all the values."""
    state['response'] = f"Hi {state['name']}, Sum of input list is {sum(state['values'])}."
    return state

In [ ]:
# Step 3: Define a Graph

from langgraph.graph import StateGraph, START, END

graph = StateGraph(AgentState)

graph.add_node("node", processor_node)

graph.add_edge(START, "node")
graph.add_edge("node", END)

In [ ]:
# Step 4: Compile the graph

app = graph.compile()

In [ ]:
# Step 5: Invoke
app.invoke({"name": "ThatAIGuy", "values": [5, -1, 0, 2, 9]})

In [ ]:
# Visualize the Graph

from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

## **Limitations of Initializing State with Pydantic**


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

# Define a State
class PydanticTemperatureState(BaseModel):
    celsius: float      # Input
    fahrenheit: Optional[float] = Field(default=None)   # Output with optional default

In [ ]:
def celsius2fahrenheit_converter(state: PydanticTemperatureState) -> PydanticTemperatureState:
    # 1. READ the input
    c_temp = state.celsius
    
    # 2. CALCULATE
    # Formula: (C * 9/5) + 32
    f_temp = (c_temp * 1.8) + 32

    # 3. WRITE the update
    state.fahrenheit = round(f_temp, 2)
    
    # 3. WRITE the update
    return state

In [ ]:
from langgraph.graph import StateGraph, START, END

pydantic_graph = StateGraph(PydanticTemperatureState)

# Add Nodes
pydantic_graph.add_node("c2f_node", celsius2fahrenheit_converter)

# Add Edges
pydantic_graph.add_edge(START, "c2f_node")
pydantic_graph.add_edge("c2f_node", END)

In [ ]:
pydantic_workflow = pydantic_graph.compile()

In [ ]:
pydantic_workflow.invoke({"celsius": 5})

## **Summary**
